In [1]:
import datetime as dt

import numpy as np
import pandas as pd

from QuantStudio.Tools.Visualization import qs_help

In [2]:
import logging
import warnings
warnings.filterwarnings('ignore')

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

# BaoStockDB

`BaoStockDB` 是基于 [BaoStock](http://baostock.com/baostock/) 在线 API 构建的因子库，主要用于**演示和测试**。通过 HTTP 请求实时获取数据，无需本地数据库。

> 注意：BaoStockDB 仅实现了 BaoStock API 的部分接口，不适合生产环境使用。可用于快速验证因子逻辑或编写示例代码。

## 连接参数

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `UserID` | `"anonymous"` | BaoStock 登录用户 ID |
| `Pwd` | `"123456"` | BaoStock 登录密码 |

通常无需配置文件，使用默认匿名账号即可。如需自定义，配置文件为 `~/QuantStudioConfig/BaoStockDBConfig.json`。

In [3]:
# 创建因子库对象并 connect（需要联网）
from QuantStudio.Factor.BaoStockDB import BaoStockDB

FDB = BaoStockDB().connect()
print(qs_help(FDB))

2026-08-27 16:14:17,818 | QS | WARNING : 找不到配置文件: C:\Users\lenovo\QuantStudioConfig\BaoStockDBConfig.json


login success!
类型: BaoStockDB
模块: QuantStudio.Factor.BaoStockDB
QS 对象类型: 因子库
QS 对象名称: BaoStockDB
QSID: c71ceadddf186b11a604aabc571d7e357ec1502375d96d737205cb2cc8c5dc9d
参数集:
    * Name(名称): <class 'str'>, 默认值 'BaoStockDB', 当前取值: 'BaoStockDB'
说明文档:
    基于 BaoStock 的因子库
    API: http://baostock.com/baostock/
    库配置信息文件在 QuantStudio 包目录下 Resource 目录下的 BaoStockDBInfo.xlsx, 记录了相关配置信息


In [4]:
# 可用的因子表
print(FDB.TableNames)

['A股K线数据', '行业分类']


# 时点与 ID 获取

BaoStockDB 提供了基本的交易日和股票代码查询。

In [5]:
# 获取交易日序列
DTs = FDB.getTradeDay(start_date=dt.datetime(2022, 1, 1), end_date=dt.datetime(2022, 1, 20))
print(DTs[:5])

login success!
[Timestamp('2022-01-04 00:00:00'), Timestamp('2022-01-05 00:00:00'), Timestamp('2022-01-06 00:00:00'), Timestamp('2022-01-07 00:00:00'), Timestamp('2022-01-10 00:00:00')]


In [6]:
# 获取当前在市的全体 A 股
IDs = FDB.getStockID()
print(IDs[:5])

login success!
['000001.SZ', '000002.SZ', '000006.SZ', '000007.SZ', '000008.SZ']


# 因子表

BaoStockDB 支持两种因子表类型，对应 BaoStock 的两种 API 模式：

| 类型 | API 模式 | 说明 |
|------|----------|------|
| `DTTable` | 逐时点查询 | 每个交易日发一次 API 请求 |
| `DTRangeTable` | 时间区间查询 | 一次请求获取整个日期范围的数据 |

两种类型均支持 `LookBack` 参数（缺失填充回溯天数）和 `APIArgs` 参数（直接透传给 BaoStock API）。

In [7]:
# 获取因子表 — A股K线数据（DTRangeTable）
FT = FDB.getTable("A股K线数据", args={"LookBack": 0})
print(qs_help(FT))

类型: _DTRangeTable
模块: QuantStudio.Factor.BaoStockDB
QS 对象类型: 计算节点-因子表
QS 对象名称: A股K线数据
QSID: 271ac426bc62996304149496cab18ce0cf189810dc9a52b059b492c0998f0384
参数集:
    * Name(名称): <class 'str'>, 默认值 'Node', 当前取值: 'A股K线数据'
    * TableType(因子表类型): typing.Literal['DTRangeTable'], 默认值 'DTRangeTable', 当前取值: 'DTRangeTable'
    * LookBack(回溯天数): <class 'int'>, 默认值 0, 当前取值: 0
说明文档:
    BaoStockDB 库中基于取时间区间数据 API 的因子表


In [8]:
# 因子列表
print(FT.FactorNames)

['open', 'high', 'low', 'close', 'preclose', 'volume', 'amount', 'adjustflag', 'turn', 'tradestatus', 'pctChg', 'peTTM', 'pbMRQ', 'psTTM', 'pcfNcfTTM', 'isST']


In [9]:
# 读取因子表数据
DTs = [dt.datetime(2025, 1, 1) + dt.timedelta(i) for i in range(5)]
IDs = ["000001.SZ", "000002.SZ"]

Data = FT.readData(factor_names=["open", "close"], ids=IDs, dts=DTs)
print("因子表数据 (Panel):")
print(Data)

因子表数据 (Panel):
<class 'QuantStudio.Core.QSObject.Panel'>
Dimensions: 2 (items) x 5 (major_axis) x 2 (minor_axis)
Items axis: open to close
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-05 00:00:00
Minor_axis axis: 000001.SZ to 000002.SZ


# 因子

通过 `FT.getFactor()` 获取的因子对象与其他因子库中的因子完全一致，同样支持运算符重载和衍生因子。详见 **[基本框架](基本框架.ipynb)** 和 **[因子开发](因子开发.ipynb)**。

In [ ]:
# 获取因子并读取数据
F = FT.getFactor("close")

DTs = [dt.datetime(2025, 1, 2), dt.datetime(2025, 1, 3)]
Data = F.readData(ids=["000001.SZ", "000002.SZ"], dts=DTs)
print(Data)